# ADNI Deployment — Train All 4 Combos + Export Artifacts


### Before running

1. **Accelerator → GPU** (Settings, right sidebar). Training 4 models on CPU would be impractically slow.
2. **Internet → On** (Settings, right sidebar). Needed for `git clone`, `pip install`, and the final HF Hub upload.
3. **Add Data → search "Augmented Alzheimer MRI Dataset" (uraninjo) → Add.**
4. **Add-ons → Secrets → add `HF_TOKEN`** (a Hugging Face token with *write* access), if you want this notebook to
   upload the trained artifacts for you at the end. Not required for the training cells themselves.


In [ ]:
import os

REPO_URL = "https://github.com/MeetRVyas/adni_code.git"
REPO_DIR = "/kaggle/working/adni_code"

%cd /kaggle/working
if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f"{REPO_DIR} already exists -- reusing it (delete the folder first for a fresh clone).")

%cd {REPO_DIR}
!pwd

In [ ]:
# Kaggle already ships torch/torchvision matched to its GPU + CUDA version,
# so this deliberately does NOT force-reinstall those (no version pin on
# torch/torchvision in requirements.txt, so pip will leave the existing
# install alone).
!pip install -q -r requirements.txt

# onnxruntime-openvino replaces plain onnxruntime (same import name) and is
# what benchmark_runtimes.py needs in order to compare the CPU and OpenVINO
# execution providers -- remove any existing onnxruntime install first so
# the two don't silently conflict.
!pip uninstall -y -q onnxruntime onnxruntime-gpu 2>/dev/null || true
!pip install -q onnx onnxscript onnxruntime-openvino

print("\nDependencies installed.")


In [ ]:
# Every training/export/manifest/benchmark script's --data_dir defaults to
# module.config.DATA_DIR, which is the bare relative name "OriginalDataset"
# (this repo's existing convention: train on the ORIGINAL, non-augmented
# images specifically, to avoid near-duplicate augmented copies of the same
# source image leaking across the train/test split). Symlinking that exact
# name at the repo root means all four combos load data identically without
# needing a --data_dir override on any of them.

KAGGLE_DATASET_ROOT = "/kaggle/input/augmented-alzheimer-mri-dataset"
print("Contents of the attached dataset:")
for entry in sorted(os.listdir(KAGGLE_DATASET_ROOT)):
    print(" ", entry)

SOURCE = f"{KAGGLE_DATASET_ROOT}/OriginalDataset"
assert os.path.isdir(SOURCE), (
    f"{SOURCE} not found -- check the listing above: if the original-images subfolder "
    f"has a different name, set SOURCE to that instead and re-run this cell."
)

LINK_NAME = "OriginalDataset"
if os.path.islink(LINK_NAME):
    os.remove(LINK_NAME)
elif os.path.exists(LINK_NAME):
    raise FileExistsError(f"./{LINK_NAME} already exists and isn't a symlink -- remove it manually first.")
os.symlink(SOURCE, LINK_NAME)

print(f"\nLinked ./{LINK_NAME} -> {SOURCE}")
print("Classes:", sorted(os.listdir(LINK_NAME)))


In [ ]:
# QUICK_TEST=True runs a fast (1 epoch / 2 folds) pass over combos 2-4, just
# to prove the whole downstream pipeline (manifest -> export -> benchmark)
# works on THIS session before committing several real GPU-hours. Note:
# train_swin.py (combo #1, the one script this notebook doesn't modify)
# has no --epochs/--nfolds flag of its own -- it always uses
# module/config.py's configured values. If you want combo #1 included in a
# fast smoke test too, temporarily lower EPOCHS/NFOLDS in module/config.py
# yourself, then restore them before the real run.
QUICK_TEST = False

def extra_args():
    args = []
    if QUICK_TEST:
        args += ["--epochs", "3", "--nfolds", "2"]
    else :
        args += ["--epochs", "100"]
    return args

# ── HF Hub target for Step 3's upload ───────────────────────────────────
HF_REPO_ID = "your-username/adni-portfolio-models"   # <-- change this

try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle secrets.")
except Exception as e:
    HF_TOKEN = None
    print(f"HF_TOKEN not available yet ({type(e).__name__}). Fine for now -- you only "
          f"need it for the upload cell in Step 3. Add it under Add-ons -> Secrets first.")


## Step 1 — Train all 4 combos

In [ ]:
import subprocess, time

def run(cmd):
    """Streams output live; raises on failure. A step failing silently here
    would mean every later step runs against an incomplete or stale set of
    artifacts without anyone noticing -- so this never swallows a bad exit code."""
    print(f"$ {' '.join(cmd)}\n")
    t0 = time.time()
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end="")
    process.wait()
    elapsed = time.time() - t0
    if process.returncode != 0:
        raise RuntimeError(f"Command failed (exit {process.returncode}) after {elapsed/60:.1f} min: {' '.join(cmd)}")
    print(f"\n[done in {elapsed/60:.1f} min]")


In [ ]:
# Combo #1: Swin-B + Progressive. --no_bda is mandatory here, not optional --
# it forces the same plain-FullDataset data loading the other 3 combos use
# (the big-data/MLflow pipeline this repo also supports is a separate,
# out-of-scope showcase project -- see PROJECT_NOTES.md). Without --no_bda,
# combo #1 could end up evaluated on a different train/test split than
# combos 2-4, which would make the leaderboard/confusion-matrix comparison
# in the app meaningless.
run(["python", "train_swin.py"] + extra_args())


In [ ]:
# Combo #2: EfficientNet-B4 + Progressive.
# Depends on the §9 fix already being present in
# module/classifiers/progressive_classifier.py (get_efficientnet_groups'
# coverage-check safeguard) -- confirmed present by the verification cell above.
run(["python", "train_efficientnet.py"] + extra_args())


In [ ]:
# Combo #3: ViT-B + Evidential (Dirichlet uncertainty).
run(["python", "train_vit_evidential.py"] + extra_args())


In [ ]:
# Combo #4: ResNeXt50 + Baseline (deliberate control group).
run(["python", "train_resnext_baseline.py"] + extra_args())


## Step 2 — Post-training artifacts (manifest, ONNX export, benchmarking)

In [ ]:
# Re-evaluates each checkpoint on its held-out test split and writes
# saved_models/manifest.json + results.json.
run(["python", "generate_manifest.py"])


In [ ]:
# PyTorch -> ONNX (fp32) -> INT8 quantized, for all 4 combos, with a
# built-in numerical-parity self-check on the fp32 export.
run(["python", "export_onnx.py"])


In [ ]:
# Validates INT8 recall parity against fp32 (falls back to fp32 if INT8
# loses too much recall) and times CPU vs OpenVINO execution providers,
# then writes both decisions back into manifest.json.
run(["python", "benchmark_runtimes.py"])


In [ ]:
import json
import pandas as pd

manifest = json.load(open("saved_models/manifest.json"))
results = json.load(open("saved_models/results.json"))

rows = []
for combo in manifest["combos"]:
    m = results["combos"].get(combo["combo_id"], {})
    rows.append({
        "combo": combo["display_name"],
        "accuracy": m.get("accuracy"),
        "recall": m.get("recall"),
        "precision": m.get("precision"),
        "f1": m.get("f1"),
        "precision_selected": combo["selected_precision"],
        "preferred_ep": combo["preferred_execution_provider"],
    })

pd.DataFrame(rows).sort_values("recall", ascending=False).reset_index(drop=True)


## Step 3 — Upload to Hugging Face Hub

Uploads `saved_models/` (weights, ONNX files, `manifest.json`, `results.json`) to the HF Hub repo the
serving app will download from at startup (`HF_REPO_ID` env var on the Space — see `serving_app/`'s
`PROJECT_NOTES.md` §5/§6.3).


In [ ]:
from huggingface_hub import HfApi

assert HF_TOKEN, "No HF_TOKEN available -- add it as a Kaggle secret (Add-ons -> Secrets) and re-run the config cell above."
assert HF_REPO_ID != "MeetV16/neuroscan-models", "Set HF_REPO_ID (a few cells up) to your actual HF Hub repo first."

api = HfApi()
api.create_repo(repo_id=HF_REPO_ID, repo_type="model", token=HF_TOKEN, exist_ok=True)

api.upload_folder(
    folder_path="saved_models",
    repo_id=HF_REPO_ID,
    repo_type="model",
    token=HF_TOKEN,
    commit_message="Training run from Kaggle notebook",
)

print(f"\nUploaded. View at: https://huggingface.co/{HF_REPO_ID}")


## Done

All 4 combos are trained, exported, benchmarked, and uploaded. What's left is deployment only:

1. Set `HF_REPO_ID` (and `HF_TOKEN` if the repo is private) as secrets/variables on your HF Space.
2. Push `Dockerfile`, `app/`, `static/`, `requirements.txt` (the serving one), and `module/` (with this same
   patch applied) from `serving_app/` to the Space repo.
3. First deploy — watch the build logs. The two things most likely to need a tweak on a real build
   (couldn't be verified from the sandbox that built this): whether `libglib2.0-0` alone is enough for
   `opencv-python-headless`, and whether `onnxruntime-openvino` needs any extra system library on the
   Spaces base image.

See `serving_app/`'s `PROJECT_NOTES.md` for the full deployment checklist.
